In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.datasets import Planetoid

# Load a sample dataset (Cora citation network)
dataset = Planetoid(root="/tmp/Cora", name="Cora")
data = dataset[0]

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1):
        super().__init__()
        # First GAT layer
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        # Second GAT layer
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1)

    def forward(self, x, edge_index):
        print(x.shape)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        # x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# Model, optimizer, loss
model = GAT(
    in_channels=dataset.num_node_features,
    hidden_channels=8,
    out_channels=dataset.num_classes,
    heads=8
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

# Testing
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum()
        acc = int(correct) / int(mask.sum())
        accs.append(acc)
    return accs

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}")


torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
torch.Size([2708, 1433])
